# 05. Maximum A Posteriori (MAP) & Regularization

**The Bayesian connection: How placing Gaussian vs Laplace Priors on model weights mathematically derives L2 (Ridge) and L1 (Lasso) Regularization.**

---

## 1. What is Maximum A Posteriori (MAP)?

While **Maximum Likelihood Estimation (MLE)** treats parameters $\mathbf{w}$ as fixed, unknown constants, **Bayesian Estimation** treats parameters $\mathbf{w}$ as random variables with a **Prior distribution $p(\mathbf{w})$** (representing our prior beliefs before seeing the data).

Using Bayes' Theorem:
$$p(\mathbf{w} | \mathbf{X}, \mathbf{y}) = \frac{p(\mathbf{y} | \mathbf{X}, \mathbf{w}) \cdot p(\mathbf{w})}{p(\mathbf{y} | \mathbf{X})} \propto p(\mathbf{y} | \mathbf{X}, \mathbf{w}) \cdot p(\mathbf{w})$$

The **Maximum A Posteriori (MAP)** estimator chooses the weights that maximize the **Posterior distribution**:

$$\hat{\mathbf{w}}_{MAP} = \arg\max_{\mathbf{w}} \left[ \log p(\mathbf{y} | \mathbf{X}, \mathbf{w}) + \log p(\mathbf{w}) \right] = \arg\max_{\mathbf{w}} \left[ \ell(\mathbf{w}) + \log p(\mathbf{w}) \right]$$

- $\ell(\mathbf{w})$ is the standard **Data Log-Likelihood** (same as MLE).
- $\log p(\mathbf{w})$ acts as a **Regularization Penalty**!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy and Matplotlib loaded!")


---

## 2. Deriving L2 Regularization (Ridge / Weight Decay) from Gaussian Prior

Suppose we place a zero-mean **Gaussian Prior** on the weights:
$$\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \tau^2 \mathbf{I}) \implies p(\mathbf{w}) = \left(\frac{1}{\tau\sqrt{2\pi}}\right)^D \exp\left( -\frac{\|\mathbf{w}\|_2^2}{2\tau^2} \right)$$

Taking the log-prior:
$$\log p(\mathbf{w}) = \text{constant} - \frac{1}{2\tau^2} \|\mathbf{w}\|_2^2$$

Substituting into the MAP objective for Gaussian Linear Regression:
$$\hat{\mathbf{w}}_{MAP} = \arg\max_{\mathbf{w}} \left[ -\frac{1}{2\sigma^2} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2 - \frac{1}{2\tau^2} \|\mathbf{w}\|_2^2 \right]$$

Negating to turn it into a minimization problem:
$$\arg\min_{\mathbf{w}} \left[ \frac{1}{N} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \lambda \|\mathbf{w}\|_2^2 \right] \quad \text{where } \lambda = \frac{\sigma^2}{N \tau^2}$$

### Theorem:
**L2 Regularization (Ridge Regression / Weight Decay) is mathematically identical to MAP estimation with a Gaussian Prior $\mathcal{N}(0, \tau^2\mathbf{I})$!**


---

## 3. Deriving L1 Regularization (Lasso / Sparsity) from Laplace Prior

Suppose instead we place a zero-mean **Laplace Prior** on the weights:
$$p(\mathbf{w}) = \prod_{j=1}^D \frac{1}{2b} \exp\left( -\frac{|w_j|}{b} \right) = \left(\frac{1}{2b}\right)^D \exp\left( -\frac{\|\mathbf{w}\|_1}{b} \right)$$

Taking the log-prior:
$$\log p(\mathbf{w}) = \text{constant} - \frac{1}{b} \|\mathbf{w}\|_1$$

Substituting into the MAP objective:
$$\arg\min_{\mathbf{w}} \left[ \frac{1}{N} \sum_{i=1}^N (y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \lambda \|\mathbf{w}\|_1 \right] \quad \text{where } \lambda = \frac{2\sigma^2}{N b}$$

### Theorem:
**L1 Regularization (Lasso Regression) is mathematically identical to MAP estimation with a Laplace Prior!**
Because the Laplace prior has a sharp, non-differentiable peak at 0, it drives irrelevant parameters exactly to zero, performing **automatic feature selection (sparsity)**.


In [ ]:
# Visualizing Gaussian Prior vs Laplace Prior
w_vals = np.linspace(-3, 3, 300)

gaussian_prior = (1.0 / np.sqrt(2*np.pi)) * np.exp(-0.5 * w_vals**2)
laplace_prior  = 0.5 * np.exp(-np.abs(w_vals))

plt.figure(figsize=(9, 5))
plt.plot(w_vals, gaussian_prior, 'b-', linewidth=2.5, label='Gaussian Prior (L2 Penalty ~ w^2)')
plt.plot(w_vals, laplace_prior, 'r-', linewidth=2.5, label='Laplace Prior (L1 Penalty ~ |w| - Sharp Peak at 0)')
plt.title("Priors as Regularizers: Gaussian (L2) vs Laplace (L1)")
plt.xlabel("Weight Value w")
plt.ylabel("Prior Probability Density p(w)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 4. Summary Comparison: MLE vs MAP vs Full Bayes

| Method | Treatment of Weights $\mathbf{w}$ | Loss Objective | Output |
| :--- | :--- | :--- | :--- |
| **MLE** | Fixed, unknown parameter | $\min -\sum \log p(y_i \mid x_i, \mathbf{w})$ | Single point estimate $\hat{\mathbf{w}}_{MLE}$ |
| **MAP** | Random variable with Prior $p(\mathbf{w})$ | $\min -\sum \log p(y_i \mid x_i, \mathbf{w}) + \text{Penalty}(\mathbf{w})$ | Single regularized point estimate $\hat{\mathbf{w}}_{MAP}$ |
| **Full Bayesian** | Full posterior distribution | Compute $p(\mathbf{w} \mid \mathcal{D}) = \frac{p(\mathcal{D}\mid \mathbf{w})p(\mathbf{w})}{\int p(\mathcal{D}\mid \mathbf{w})p(\mathbf{w})d\mathbf{w}}$ | Full distribution over weights + Uncertainty estimation |

---

## 5. Summary & Key Takeaways

1. **MAP Estimation** combines observed data likelihood with a prior belief over model weights.
2. A **Gaussian Prior** corresponds to **$L_2$ Regularization (Ridge / Weight Decay)**, keeping weights small.
3. A **Laplace Prior** corresponds to **$L_1$ Regularization (Lasso)**, enforcing exact sparsity.
